In [2]:
import torch
import numpy as np
from sklearn.metrics import f1_score
from transformers import pipeline
from datasets import load_dataset
from huggingface_hub import login
from datasets.arrow_dataset import Dataset
from datasets.dataset_dict import DatasetDict, IterableDatasetDict
from datasets.iterable_dataset import IterableDataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForCausalLM
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

In [3]:
import transformers, torch, accelerate, huggingface_hub
print(transformers.__version__)
print(torch.__version__)
print(accelerate.__version__)
print(huggingface_hub.__version__)

5.3.0
2.10.0+cpu
1.13.0
1.6.0


In [ ]:

login(token="")

In [5]:
# Dataset id from huggingface.co/dataset
dataset_id = "argilla/synthetic-domain-text-classification"
 
# Load raw dataset
dataset = load_dataset(dataset_id, split = 'train')

dataset


Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})

In [6]:
# Prepare model labels - useful for inference
labels = dataset.features["label"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [7]:
split_dataset = dataset.train_test_split(test_size=0.1)
split_dataset['train'][0]
# {'text': 'Recently, there has been an increase in property values within the suburban areas of several cities due to improvements in infrastructure and lifestyle amenities such as parks, retail stores, and educational institutions nearby. Additionally, new housing developments are emerging, catering to different family needs with varying sizes and price ranges. These changes have influenced investment decisions for many looking to buy or sell properties.', 'label': 14}

split_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 900
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 100
    })
})

In [8]:
model_id = "mistralai/Mistral-7B-v0.1"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Ensure pad token is set (reuse eos token)
tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    prompt = f"Text: {example['text']}\nSentiment:"
    completion = f" {id2label[str(example['label'])]}"  # space is important
    full_text = prompt + completion
    return tokenizer(full_text, truncation=True)

tokenized_dataset = split_dataset.map(format_example)

# Inspect features
print(tokenized_dataset['train'][0])

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

{'text': 'In this essay, we explore the rich cultural heritage of modern musical genres, tracing their origins from classical music forms to contemporary popular styles. The interplay between traditional instruments and modern electronic elements has created a unique synthesis that has captivated audiences worldwide. Notable artists like Beyoncé and Kendrick Lamar have been at the forefront of blending different cultural influences into powerful, innovative musical pieces.', 'label': 21, 'input_ids': [1, 7379, 28747, 560, 456, 10360, 28725, 478, 11418, 272, 6708, 8932, 22597, 302, 4638, 9158, 2652, 411, 28725, 28469, 652, 26081, 477, 13378, 3427, 6967, 298, 13621, 4387, 12078, 28723, 415, 791, 1674, 1444, 7062, 17937, 304, 4638, 13176, 5176, 659, 3859, 264, 4842, 13606, 21537, 369, 659, 4286, 449, 601, 24460, 15245, 28723, 2280, 522, 10611, 737, 16015, 266, 22034, 304, 524, 416, 9450, 393, 11369, 506, 750, 438, 272, 2417, 6072, 302, 843, 2570, 1581, 8932, 26386, 778, 6787, 28725, 16827

In [9]:
def mask_prompt_labels(example):
    input_ids = example['input_ids']
    # Find index where label starts
    label_start = input_ids.index(tokenizer(" " + id2label[str(example['label'])]).input_ids[1])
    labels = [-100] * label_start + input_ids[label_start:]
    example['labels'] = labels
    return example

masked_tokenized_dataset = tokenized_dataset.map(mask_prompt_labels)

print(masked_tokenized_dataset['train'][0])

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

{'text': 'In this essay, we explore the rich cultural heritage of modern musical genres, tracing their origins from classical music forms to contemporary popular styles. The interplay between traditional instruments and modern electronic elements has created a unique synthesis that has captivated audiences worldwide. Notable artists like Beyoncé and Kendrick Lamar have been at the forefront of blending different cultural influences into powerful, innovative musical pieces.', 'label': 21, 'input_ids': [1, 7379, 28747, 560, 456, 10360, 28725, 478, 11418, 272, 6708, 8932, 22597, 302, 4638, 9158, 2652, 411, 28725, 28469, 652, 26081, 477, 13378, 3427, 6967, 298, 13621, 4387, 12078, 28723, 415, 791, 1674, 1444, 7062, 17937, 304, 4638, 13176, 5176, 659, 3859, 264, 4842, 13606, 21537, 369, 659, 4286, 449, 601, 24460, 15245, 28723, 2280, 522, 10611, 737, 16015, 266, 22034, 304, 524, 416, 9450, 393, 11369, 506, 750, 438, 272, 2417, 6072, 302, 843, 2570, 1581, 8932, 26386, 778, 6787, 28725, 16827

In [10]:
masked_tokenized_dataset['train'].features["label"].names

['business-and-industrial',
 'books-and-literature',
 'beauty-and-fitness',
 'autos-and-vehicles',
 'people-and-society',
 'sports',
 'shopping',
 'online-communities',
 'pets-and-animals',
 'internet-and-telecom',
 'home-and-garden',
 'adult',
 'science',
 'food-and-drink',
 'real-estate',
 'news',
 'jobs-and-education',
 'health',
 'hobbies-and-leisure',
 'games',
 'computers-and-electronics',
 'arts-and-entertainment',
 'travel-and-transportation',
 'finance',
 'law-and-government',
 'sensitive-subjects']

In [11]:

# Download the model from huggingface.co/models
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    dtype="auto"
)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [12]:
lora_config = LoraConfig(
    r=16,                 # rank of LoRA update
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],  # Qwen attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # verify only LoRA params are trainable

trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


In [13]:
# Metric helper method: generative classification via label likelihood
def compute_metrics(eval_pred):
    texts = split_dataset['test']['text']
    true_labels = split_dataset['test']['label']
    label_texts = dataset.features['label'].names

    predictions = []

    model.eval()
    with torch.no_grad():
        for text in texts:
            # Same prompt format used in training examples.
            prompt = f"Text: {text}\\nSentiment:"
            prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids.to(model.device)

            label_scores = []
            for label in label_texts:
                completion = " " + label
                completion_ids = tokenizer(
                    completion,
                    return_tensors="pt",
                    add_special_tokens=False,
                ).input_ids.to(model.device)

                # Concatenate prompt + candidate label and score only candidate tokens.
                input_ids = torch.cat([prompt_ids, completion_ids], dim=1)
                outputs = model(input_ids=input_ids)
                log_probs = torch.log_softmax(outputs.logits, dim=-1)

                start = prompt_ids.shape[1]
                token_log_likelihood = 0.0
                for i in range(start, input_ids.shape[1]):
                    token_id = input_ids[0, i]
                    token_log_likelihood += log_probs[0, i - 1, token_id].item()

                # Normalize by candidate length so longer labels are not unfairly penalized.
                normalized_score = token_log_likelihood / completion_ids.shape[1]
                label_scores.append(normalized_score)

            pred_label = int(np.argmax(label_scores))
            predictions.append(pred_label)

    score = f1_score(true_labels, predictions, average="weighted")
    return {"f1": score}

In [14]:
# Define training args
training_args = TrainingArguments(
    output_dir="Generative-domain-classifier",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=5,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    bf16=False,
    fp16=False,
    optim="adamw_torch",

    logging_strategy="steps",
    logging_steps=1,          # show frequent updates
    logging_first_step=True,  # print from first step
    disable_tqdm=False,       # force progress bar
    report_to="none",         # avoid integrations hanging output

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",

    push_to_hub=False,        # disable for local debugging
)
 
# Create a Trainer instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=masked_tokenized_dataset["train"],
    eval_dataset=masked_tokenized_dataset["test"],
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.train()